# Phase A - Step 2: PyTorch Dataset Class

This notebook creates the PyTorch `Dataset` to wrap the preprocessed images and masks generated in Step A1. It heavily utilizes **Albumentations** for robust, on-the-fly data augmentation which is crucial for segmentation performance.

**Key Highlights:**
- Custom PyTorch `RetinalDataset` class.
- `Albumentations` transform pipelines (Geometric + Color).
- ImageNet Normalization and conversion to PyTorch Tensors.
- Visual sanity checking by grabbing a DataLoader batch and de-normalizing.

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
import zipfile


zip_path = "dataset_preprocessed_V2.zip"     # Change this to your actual zip file name
extract_dir = "dataset"           # Where to extract your files

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

## 1. Defining the PyTorch Dataset
The generic Dataset will load an image and its corresponding mask, then pass both simultaneously into `albumentations` so spatial transforms (like flipping/rotating) are precisely applied to both.

In [ ]:
class RetinalDataset(Dataset):
    def __init__(self, csv_file, base_dir, transform=None):
        """
        Args:
            csv_file (string): Path to the csv file with target paths (train/val/test split).
            base_dir (string): Base directory holding the preprocessed files.
            transform (callable, optional): Optional Albumentations transform Pipeline.
        """
        self.df = pd.read_csv(csv_file)
        self.base_dir = base_dir
        self.transform = transform

        # For lesion segmentation (A7-A9), we can expand this to load multiple masks!
        # For now, it maps solely to 'vessel_path'.

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        # Construct full paths
        img_path = os.path.join(self.base_dir, row['img_path'])
        mask_path = os.path.join(self.base_dir, row['vessel_path'])

        # Load Image (Albumentations expects RGB numpy arrays)
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        # Load Vessel Mask (Grayscale)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        # Binarize and convert to float32 (values either 0.0 or 1.0)
        mask = (mask > 127).astype(np.float32)

        # Apply Augmentations (if any)
        if self.transform is not None:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']

        # Albumentations ToTensorV2 returns masks without channel dim if they are 2D.
        # We need PyTorch target shape: (1, Height, Width) for binary cross entropy.
        if isinstance(mask, torch.Tensor):
            mask = mask.unsqueeze(0)
        else:
            mask = torch.from_numpy(mask).unsqueeze(0)

        return image, mask

## 2. Augmentation Pipelines
Geometric transforms work incredibly well on the retina since biological orientations are strictly arbitrary. We also include ImageNet standardization required for CNN/ViT backbones.

In [ ]:
def get_train_transforms():
    return A.Compose([
        # Geometric Variations
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1, rotate_limit=45, p=0.6),
        A.GridDistortion(p=0.2),

        # Color/Illumination Variations (Important for different fundus cameras)
        A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=0.4),

        # Normalization (Pretrained backbone expectations)
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

def get_val_transforms():
    return A.Compose([
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])

## 3. Visualization and DataLoader Testing
Grab a batch, reverse the standardizations (to render valid images) and display!

In [ ]:
def denormalize(tensor, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    """Reverses the ImageNet Normalization so we can plt.imshow correctly."""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    tensor = tensor * std + mean
    tensor = torch.clamp(tensor, 0, 1)
    return tensor.permute(1, 2, 0).numpy()

if __name__ == '__main__':
    # Standard location expected from notebook 01
    BASE_DIR = 'dataset'
    TRAIN_CSV = os.path.join(BASE_DIR, 'train_split.csv')

    if os.path.exists(TRAIN_CSV):
        print("Successfully found training split!")
        # Create Dataset & DataLoader
        train_dataset = RetinalDataset(csv_file=TRAIN_CSV, base_dir=BASE_DIR, transform=get_train_transforms())
        train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)

        # Extract first batch
        images, masks = next(iter(train_loader))
        print(f"Batched Images Shape: {images.shape}") # should be [4, 3, 512, 512]
        print(f"Batched Masks  Shape: {masks.shape}")  # should be [4, 1, 512, 512]

        # Plotting
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        for i in range(4):
            img_np = denormalize(images[i])

            axes[0, i].imshow(img_np)
            axes[0, i].set_title(f"Augmented Image {i+1}")
            axes[0, i].axis("off")

            axes[1, i].imshow(masks[i].squeeze(), cmap='gray')
            axes[1, i].set_title(f"Augmented Mask {i+1}")
            axes[1, i].axis("off")

        plt.tight_layout()
        plt.show()
    else:
        print(f"Data not found at {TRAIN_CSV}. Make sure to completely run 01_preprocessing.ipynb first!")